# 第2回 機械学習とは — 実習ノートブック

**このノートブックは第2回だけで完結します。** プログラミングの経験は要りません。

やること:
1. 60人の身長・体重から予測の式を作る
2. 訓練データとテストデータに分ける
3. わざと「丸暗記」させて、何が起きるか見る

所要時間:実習 約20分 / 課題 約40分

## 0. 準備

In [ ]:
# 準備(このセルを最初に1回だけ実行)
!pip install -q japanize-matplotlib
import numpy as np, matplotlib.pyplot as plt, japanize_matplotlib
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
plt.rcParams["font.size"] = 12
print("準備できました")

## 1. データを見る

60人ぶんの身長と体重です。ここから「身長を入れると体重が出る式」を作ります。

In [ ]:
# ===== データを用意する(60人の身長と体重)=====
乱数 = np.random.default_rng(2)
n = 60
身長 = 乱数.normal(165, 8, n).round(1)
体重 = (0.75 * 身長 - 62 + 乱数.normal(0, 4.0, n)).round(1)

print(f"{n}人ぶんのデータができました")
print(f"  身長 平均 {身長.mean():.1f}cm(範囲 {身長.min():.1f}〜{身長.max():.1f})")
print(f"  体重 平均 {体重.mean():.1f}kg(範囲 {体重.min():.1f}〜{体重.max():.1f})")

plt.figure(figsize=(7, 4.5))
plt.scatter(身長, 体重, color="#3A46A8", alpha=.7)
plt.xlabel("身長(cm)"); plt.ylabel("体重(kg)"); plt.title("60人の身長と体重")
plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## 2. 訓練データとテストデータに分ける

**これが機械学習の基本作法です。** 全部のデータで学習させてしまうと、性能を測る手段がなくなります。

In [ ]:
# ===== 訓練データとテストデータに分ける =====
X = 身長.reshape(-1, 1)          # sklearn は「1行1人」の形を求めるので縦に並べる
訓練X, テストX, 訓練y, テストy = train_test_split(X, 体重, test_size=0.3, random_state=2)

print(f"訓練データ {len(訓練X)}人  ← ここだけ見せて学習させる")
print(f"テストデータ {len(テストX)}人  ← 最後まで隠しておき、評価にだけ使う")

## 3. モデルを作る — たった1行

`fit()` が「学習」です。点の集まりに、いちばん近い直線を1本引いています。

In [ ]:
# ===== ① 直線を引くモデル =====
直線 = LinearRegression()
直線.fit(訓練X, 訓練y)            # ← これが「学習」

a, b = 直線.coef_[0], 直線.intercept_
print(f"できた式:  体重 = {a:.2f} × 身長 − {-b:.1f}\n")
print(f"170cm の人の予測 → {直線.predict([[170]])[0]:.1f} kg")
print(f"160cm の人の予測 → {直線.predict([[160]])[0]:.1f} kg\n")
print(f"訓練データでの平均誤差   {mean_absolute_error(訓練y, 直線.predict(訓練X)):.2f} kg")
print(f"テストデータでの平均誤差 {mean_absolute_error(テストy, 直線.predict(テストX)):.2f} kg")

## 4. わざと「丸暗記」させてみる

訓練データの点を1つずつ覚えるモデルを作ります。**訓練データでの成績は完璧になります。**
問題は、そのあとです。

In [ ]:
# ===== ② 点を丸暗記するモデル =====
# 決定木に制限をかけないと、訓練データの点を1つずつ覚えてしまいます。
丸暗記 = DecisionTreeRegressor(random_state=0)
丸暗記.fit(訓練X, 訓練y)

print("【訓練データでの誤差】")
print(f"  直線モデル   {mean_absolute_error(訓練y, 直線.predict(訓練X)):.2f} kg")
print(f"  丸暗記モデル {mean_absolute_error(訓練y, 丸暗記.predict(訓練X)):.2f} kg  ← ほぼ完璧!\n")
print("【テストデータでの誤差】= 見たことのない人で試すと")
print(f"  直線モデル   {mean_absolute_error(テストy, 直線.predict(テストX)):.2f} kg")
print(f"  丸暗記モデル {mean_absolute_error(テストy, 丸暗記.predict(テストX)):.2f} kg  ← 逆転!\n")
print("訓練データだけ見ていたら、丸暗記モデルを選んでしまいます。")

In [ ]:
# ===== 2つのモデルを図で見比べる(課題2用)=====
細かい身長 = np.linspace(身長.min(), 身長.max(), 400).reshape(-1, 1)

plt.figure(figsize=(9, 5))
plt.scatter(訓練X, 訓練y, color="#3A46A8", alpha=.7, label="訓練データ(42人)")
plt.scatter(テストX, テストy, color="#E08A2B", marker="^", s=70, label="テストデータ(18人)")
plt.plot(細かい身長, 直線.predict(細かい身長), color="#3A46A8", lw=2.5, label="① 直線モデル")
plt.plot(細かい身長, 丸暗記.predict(細かい身長), color="#C0392B", lw=1.8, ls="--", label="② 丸暗記モデル")
plt.xlabel("身長(cm)"); plt.ylabel("体重(kg)")
plt.title("直線モデルと丸暗記モデル")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

print("丸暗記モデル(赤い破線)は、訓練データの点を通るために階段状にガタガタ動きます。")
print("点と点のあいだには、何の根拠もありません。だから新しい人(▲)には答えられません。")

## 5. 訓練データの割合を変える(演習・問2)

In [ ]:
# ===== 演習・問2:訓練データの割合を変える =====
print(f"{'訓練の割合':>12}{'訓練の人数':>12}{'訓練誤差':>12}{'テスト誤差':>12}")
for 割合 in [0.9, 0.7, 0.5, 0.3]:
    trX, teX, trY, teY = train_test_split(X, 体重, train_size=割合, random_state=2)
    m = LinearRegression().fit(trX, trY)
    print(f"{割合:>11.0%}{len(trX):>12}"
          f"{mean_absolute_error(trY, m.predict(trX)):>11.2f}kg"
          f"{mean_absolute_error(teY, m.predict(teX)):>11.2f}kg")

print("\n【考えてみてください】")
print("訓練データを100%にすればもっと良くなるのでは? 何が問題でしょうか。")

## 6. 自分で試す

書きかえて実行してください。壊しても大丈夫です。

In [ ]:
# ===== 自分で試す =====
# (a) 自分の身長を入れてみる
私の身長 = 170          # ← 書きかえてください
私の体重 = 60           # ← 書きかえてください
予測 = 直線.predict([[私の身長]])[0]
print(f"身長 {私の身長}cm → 予測 {予測:.1f}kg / 実際 {私の体重}kg / ずれ {abs(予測-私の体重):.1f}kg")
print("→ 数kgずれるのが普通です。身長だけでは、骨格も筋肉量も年齢も分からないからです。\n")

# (b) 入力を増やすとどうなるか(身長に加えて「年齢」を足してみる)
年齢 = 乱数.integers(18, 60, n)
体重2 = 体重 + (年齢 - 30) * 0.15        # 年齢でも少し変わる、という架空の設定
X2 = np.column_stack([身長, 年齢])
trX, teX, trY, teY = train_test_split(X2, 体重2, test_size=0.3, random_state=2)
m2 = LinearRegression().fit(trX, trY)
trX1, teX1, trY1, teY1 = train_test_split(X, 体重2, test_size=0.3, random_state=2)
m1 = LinearRegression().fit(trX1, trY1)
print(f"身長だけ      → テスト誤差 {mean_absolute_error(teY1, m1.predict(teX1)):.2f} kg")
print(f"身長 + 年齢   → テスト誤差 {mean_absolute_error(teY, m2.predict(teX)):.2f} kg")
print("→ 効く入力を足すと精度が上がります。「入力に何を選ぶか」は人間の仕事です。")

## 7. 提出課題(manaba)

**締切:次回講義の前日 23:59**

### 課題1:身のまわりの「予測したいこと」で三点セットを設計する
授業で扱った例(体重・迷惑メール・マンション価格)**以外**から選んでください。

- (a) 何を予測しますか
- (b) 入力は何にしますか(3つ以上)
- (c) 正解の札はどこから手に入れますか ← ここが実務でいちばん難しいところです
- (d) 損失は何にしますか(左右で痛さが違うなら、そう書いてください)
- (e) その予測を実際に使うとしたら、誰がどう困る可能性がありますか

### 課題2:ノートブックを動かして報告する
- (a) 直線モデルと丸暗記モデルの、訓練誤差とテスト誤差の表
- (b) セクション4の散布図(2つのモデルを重ねた図)
- (c) 図を見て、丸暗記モデルが何をしているかを自分の言葉で説明する(3行以上)

### 課題3:次の主張を批判する(300字以上)

> 「我々のAIは、社内の過去5年分のデータで学習させ、精度98%を達成しました。
> 人間の担当者の正答率は85%ですから、この業務はAIに任せるべきです。」

**数字が嘘だと言いたいわけではありません。** 「その98%は何を測った数字か」という角度から考えてください。

考える材料:
- 演習・問3で練習した質問
- 人間の85%と、同じ条件で比べていますか
- 「過去5年分」で学んだモデルは、これからも通用しますか
- 間違えたとき、誰が責任を持ちますか

**加点**:「では、どういう条件がそろえば任せてよいか」まで書けると評価が上がります。